# Chempiler — Analysis Walkthrough

**Chempiler** is a Python package for chemical perception and analysis of reactive molecular dynamics (MD) trajectories.
Unlike conventional MD analysis tools, chempiler rebuilds molecular topology from scratch for every frame,
making it correct for reactive force-field simulations (e.g. ReaxFF) where bonds break and form throughout the run.

This notebook walks through the main capabilities:

| Section | What it covers |
|---|---|
| 1 | Loading a trajectory and HDF5 caching |
| 2 | Inspecting the chemical composition |
| 3 | Trajectory segmentation |
| 4 | Radial distribution function (RDF) |
| 5 | Atom-hop tracking (proton transfer) |
| 6 | Ligand-exchange and coordination dynamics |
| 7 | Block-averaging statistics |
| 8 | Custom tracking with the low-level `Tracker` API |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

---
## 1 — Loading a trajectory

`ChempilerTrajectory` accepts any ASE-readable file format (`.traj`, `.xyz`, LAMMPS dump, etc.).
Calling `.build()` performs chemical perception on every frame: atoms are grouped into molecules
by distance-based bond detection using scaled ASE covalent radii.

### HDF5 caching

Perception is the most expensive step — rebuilding all molecules for 20 000 frames takes seconds to minutes
depending on system size. Pass a `cache_file` path and the result is serialised to HDF5 on the first run
and reloaded instantly on every subsequent run. The cache is invalidated automatically if any perception
parameter changes.

In [ ]:
from chempiler import ChempilerTrajectory

traj = ChempilerTrajectory(
    "tests/39_water_OH.traj",
    mode="molecular",       # detect covalent bonds
    covalent_scale=1.0,     # exact ASE natural cutoffs
)

traj.build(cache_file="tests/cache_file.h5")

frames = traj.frames
print(f"Loaded {len(frames)} frames")
print(f"Atoms per frame : {len(frames[0].atoms)}")
print(f"Molecules in frame 0 : {len(frames[0].molecules)}")

The `[Chempiler] loaded cache` message confirms that the HDF5 file was found and used.
Running the cell a second time will always load from cache — the build step is skipped entirely.

Each `Frame` object exposes:
- `frame.atoms` — the raw ASE `Atoms` object (positions, cell, PBC)
- `frame.molecules` — list of atom-index lists, one per molecule
- `frame.formulas` — list of formula strings parallel to `molecules`
- `frame.symbols` — element symbol per atom
- `frame.atom_to_mol` — int32 array mapping each atom index to its molecule index

---
## 2 — Chemical composition

`traj.summary()` returns how many times each molecular formula appears across all frames.
In a stable trajectory this is roughly `n_frames × count_per_frame` for each species;
large imbalances between formulas signal reaction events.

In [ ]:
summary = traj.summary()

# Sort by count so the dominant species appear first
for formula, count in sorted(summary.items(), key=lambda x: -x[1]):
    avg_per_frame = count / len(frames)
    print(f"  {formula:<8}  total={count:>8}   avg/frame={avg_per_frame:.2f}")

**Interpretation:** `H2O` will be the dominant species. Smaller counts of `HO` (hydroxide) or `H3O` (hydronium)
indicate transient proton-transfer intermediates. Because molecular identity is rebuilt each frame,
any frame where an O atom briefly picks up a second H registers as `H3O`, and any frame where it
loses one registers as `HO`.

We can also inspect the composition of a single frame more directly:

In [ ]:
from collections import Counter

frame0 = frames[0]
counts = Counter(frame0.formulas)

print("Frame 0 composition:")
for formula, n in sorted(counts.items(), key=lambda x: -x[1]):
    print(f"  {formula}: {n}")

# Show which atoms belong to the first H2O molecule
mol_id = frame0.formula_to_mols["H2O"][0]
atom_indices = frame0.atoms_in_mol(mol_id)
print(f"\nFirst H2O molecule → atom indices {atom_indices}")
for a in atom_indices:
    print(f"  atom {a}: {frame0.symbols[a]} at {frame0.positions[a]}")

### Molecule count over time

Plotting the total number of molecules per frame is a quick sanity check:
a flat line means nothing reacted; a step change means a reaction happened.

In [ ]:
mol_counts = [len(f.molecules) for f in frames]
h2o_counts = [f.formulas.count("H2O") for f in frames]

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

axes[0].plot(mol_counts, lw=0.8, color="steelblue")
axes[0].set_ylabel("Total molecules")
axes[0].set_title("Molecular population over the trajectory")

axes[1].plot(h2o_counts, lw=0.8, color="forestgreen")
axes[1].set_ylabel("H₂O count")
axes[1].set_xlabel("Frame")

plt.tight_layout()
plt.show()

**Interpretation:** In a bulk water simulation run with a reactive force field you typically see the H₂O
count fluctuate by ±1–2 around the starting value, reflecting transient proton-transfer states where
one molecule momentarily becomes H₃O⁺ and another OH⁻. A sustained drop would indicate a genuine
dissociation event.

---
## 3 — Trajectory segmentation

For long trajectories that contain a clear reaction (e.g. an acid–base neutralisation) it is useful
to split the run into segments of roughly constant composition before computing time averages.
`segment_by_molecule_count` does this by dividing the trajectory into non-overlapping blocks and
looking for a step change in the mean molecule count between adjacent blocks.

In [ ]:
from chempiler.segmentation import segment_by_molecule_count

segments = segment_by_molecule_count(
    frames,
    block=50,       # frames averaged per block when searching for boundaries
    threshold=0.5,  # minimum change in mean molecule count to declare a new segment
)

print(f"Found {len(segments)} segment(s):")
for i, (start, end) in enumerate(segments):
    n_frames_seg = end - start
    avg_mols = np.mean(mol_counts[start:end])
    print(f"  Segment {i}: frames [{start}, {end})  ({n_frames_seg} frames)  avg mols = {avg_mols:.2f}")

In [ ]:
colors = plt.cm.Set2.colors

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(mol_counts, lw=0.8, color="0.4", zorder=2)

for i, (start, end) in enumerate(segments):
    ax.axvspan(start, end, alpha=0.25, color=colors[i % len(colors)], label=f"Segment {i}")

ax.set_xlabel("Frame")
ax.set_ylabel("Total molecules")
ax.set_title("Trajectory segments")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

**Note:** `segment_by_molecule_count` detects changes in the *total* number of molecules.
It will not detect isodesmic reactions (e.g. H₃O⁺ + OH⁻ → 2 H₂O) because those conserve
the count. For finer-grained reaction detection, see `ligand_exchange` and `coordination_dynamics` in section 6.

---
## 4 — Radial distribution function (RDF)

The RDF $g(r)$ describes how atomic density varies as a function of distance from a reference atom.
A peak at distance $r$ means that target atoms preferentially sit at that separation from the center atoms.

Chempiler computes RDFs correctly under periodic boundary conditions using ASE's `find_mic`
(minimum image convention). Selectors are resolved fresh for each frame, so changing populations
(reactions) are handled correctly.

### Selector syntax

| Selector | Meaning |
|---|---|
| `"O"` | All oxygen atoms |
| `"H"` | All hydrogen atoms |
| `{"H2O": "O"}` | O atoms that are inside an H₂O molecule |
| `{"H2O": None}` | All atoms inside H₂O molecules |

In [ ]:
# O–H RDF: the first peak is the O–H covalent bond (~1 Å),
# the second peak is the hydrogen-bond donor distance (~1.8 Å)
r_oh, g_oh = traj.rdf(center="O", target="H", rmax=4.0, dr=0.02)

# O–O RDF: the first peak at ~2.8 Å is the hydrogen-bond O–O distance
r_oo, g_oo = traj.rdf(center="O", target="O", rmax=6.0, dr=0.05)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(r_oh, g_oh, color="steelblue", lw=1.2)
axes[0].axhline(1.0, color="0.5", lw=0.8, ls="--", label="bulk limit")
axes[0].set_xlabel("r (Å)")
axes[0].set_ylabel("g(r)")
axes[0].set_title("O–H radial distribution function")
axes[0].legend()

axes[1].plot(r_oo, g_oo, color="tomato", lw=1.2)
axes[1].axhline(1.0, color="0.5", lw=0.8, ls="--", label="bulk limit")
axes[1].set_xlabel("r (Å)")
axes[1].set_ylabel("g(r)")
axes[1].set_title("O–O radial distribution function")
axes[1].legend()

plt.tight_layout()
plt.show()

**Interpretation:**
- **O–H:** The sharp peak near 1 Å is the covalent O–H bond. The broader peak around 1.8 Å corresponds
  to hydrogen-bond donated H atoms. Both peaks should sit well above 1 (the ideal-gas baseline).
- **O–O:** The peak near 2.8 Å is the characteristic hydrogen-bond O–O distance in liquid water;
  the second shell appears around 4.5 Å. $g(r) \to 1$ at large $r$ confirms the simulation is large
  enough that the correlations die out before the box boundary.

### Running coordination number

Setting `integrate=True` returns the cumulative integral $n(r) = 4\pi\rho \int_0^r g(r') r'^2 dr'$,
the average number of target atoms within distance $r$ of each center atom.
The value of $n(r)$ at the first minimum gives the mean first-shell coordination number.

In [ ]:
r_cn, n_oh = traj.rdf(center="O", target="H", rmax=4.0, dr=0.02, integrate=True)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(r_cn, n_oh, color="steelblue", lw=1.2)
ax.axhline(2.0, color="0.5", lw=0.8, ls="--", label="expected: 2 H per H₂O")
ax.set_xlabel("r (Å)")
ax.set_ylabel("n(r)")
ax.set_title("O–H running coordination number")
ax.legend()
plt.tight_layout()
plt.show()

# Read off the coordination number at the first minimum (just past the covalent-bond peak)
mask = r_cn < 1.4
cn_covalent = n_oh[mask][-1]
print(f"Average covalent O–H coordination number (r < 1.4 Å): {cn_covalent:.2f}")

**Interpretation:** The plateau near $n = 2$ at $r \approx 1.2$ Å confirms that each oxygen has on
average two covalently bonded hydrogens — consistent with a mostly H₂O population. A value slightly
below 2 would indicate a small fraction of OH⁻ anions.

---
## 5 — Atom-hop tracking

`atom_hop` tracks the transfer of a mobile species (default: `H`) between host atoms (default: `O`).
The *state* of each tracked atom is the index of the nearest host within `cutoff` Å.
A hop is recorded whenever that index changes.

The `persistence` parameter is a noise filter: the new host must be observed for at least
`persistence` consecutive frames before the hop is committed. This suppresses spurious events
that arise when a hydrogen rattles back and forth across the cutoff boundary.

In [ ]:
from chempiler.state_engine import atom_hop

result = atom_hop(
    frames,
    tracked="H",        # mobile species
    host="O",           # host species
    cutoff=1.25,        # Å — rough O–H covalent bond cutoff
    persistence=3,      # new host must hold for 3 frames
)

print(f"Total hop events : {result['n_transitions']}")
print(f"Frames in trajectory : {len(frames)}")

if result['n_transitions'] > 0:
    print("\nFirst 5 hop events (frame, H-atom, from-O, to-O):")
    for event in result['transitions'][:5]:
        frame_idx, h_idx, from_o, to_o = event
        print(f"  frame {frame_idx:>5}  H[{h_idx:>3}]  O[{from_o}] → O[{to_o}]")

In [ ]:
transitions = result['transitions']
residence = result['residence_times']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Hop events over time
if transitions:
    hop_frames = [t[0] for t in transitions]
    axes[0].hist(hop_frames, bins=50, color="steelblue", edgecolor="white", lw=0.4)
    axes[0].set_xlabel("Frame")
    axes[0].set_ylabel("Hop events")
    axes[0].set_title("Proton transfer events over time")
else:
    axes[0].text(0.5, 0.5, "No hops detected", ha="center", va="center",
                 transform=axes[0].transAxes, fontsize=13, color="0.5")
    axes[0].set_title("Proton transfer events over time")

# Residence-time distribution
axes[1].hist(residence, bins=40, color="tomato", edgecolor="white", lw=0.4)
axes[1].set_xlabel("Residence time (frames)")
axes[1].set_ylabel("Count")
axes[1].set_title("H-atom residence-time distribution")
axes[1].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.show()

if len(residence) > 0:
    print(f"Mean residence time : {residence.mean():.1f} frames")
    print(f"Max  residence time : {residence.max()} frames")

**Interpretation:**
- The **event histogram** shows when proton transfers are most frequent. A uniform distribution
  indicates diffuse, uncorrelated hopping (typical in liquid water). Bursts or gaps would point to
  collective or concerted mechanisms.
- The **residence-time distribution** is typically exponential for a Poisson process. A very short
  mean residence time (comparable to the time step) suggests the `cutoff` or `persistence` needs tuning.

### Tracking any mobile species

`atom_hop` is not limited to protons. The same function works for any mobile/host pair —
lithium hopping in a solid electrolyte, halide migration in a crystal, etc.

In [ ]:
# Example: same call for a different system (Li hopping between O sites)
# result_li = atom_hop(frames, tracked="Li", host="O", cutoff=2.2, persistence=5)

# Here we just show the call signature; uncomment for the appropriate trajectory
print("atom_hop(frames, tracked='Li', host='O', cutoff=2.2, persistence=5)")
print("Returns the same dict: {'transitions', 'n_transitions', 'residence_times'}")

---
## 6 — Ligand exchange and coordination dynamics

These two functions use the internal per-frame tracker, which re-evaluates the entity set at every
frame. This is correct for reactive systems where molecules appear or disappear mid-trajectory.

### 6a — Ligand exchange

Tracks changes in molecular *formula*. A transition from `H2O` → `HO` means that molecule lost
a hydrogen (donated it to a neighbour); `H2O` → `H3O` means it gained one.

In [ ]:
from chempiler.state_engine import ligand_exchange

lex = ligand_exchange(frames)

print(f"Ligand-exchange events: {lex['n_transitions']}")

if lex['n_transitions'] > 0:
    # Summarise which transitions actually occurred
    from collections import Counter
    transition_types = Counter(
        (from_f, to_f) for _, _, from_f, to_f in lex['transitions']
    )
    print("\nTransition types:")
    for (a, b), count in sorted(transition_types.items(), key=lambda x: -x[1]):
        print(f"  {a} → {b} : {count}")

### 6b — Coordination dynamics

Tracks changes in the *bonding environment* of each atom of a given element. The state is the
sorted tuple of neighbour element symbols inside the same molecule. A transition from `('H','H','O')`
to `('H','O')` means the O atom lost a covalent-bond partner (proton transferred away).

In [ ]:
from chempiler.state_engine import coordination_dynamics

cdyn = coordination_dynamics(frames, atom_symbol="O")

print(f"Coordination-change events: {cdyn['n_transitions']}")

if cdyn['n_transitions'] > 0:
    from collections import Counter
    env_changes = Counter(
        (from_e, to_e) for _, _, from_e, to_e in cdyn['transitions']
    )
    print("\nEnvironment transitions (top 5):")
    for (a, b), count in env_changes.most_common(5):
        print(f"  {a} → {b} : {count}")

**Interpretation:** `ligand_exchange` and `coordination_dynamics` are complementary:
- Ligand exchange counts *molecular identity* changes; it fires once per reaction per molecule.
- Coordination dynamics fires on each *atom* individually and is more sensitive to partial
  rearrangements that do not change the overall formula (e.g., bond-angle fluctuations that
  briefly break one bond while forming another).

---
## 7 — Block-averaging statistics

Trajectory data is time-correlated: adjacent frames are not independent samples. Computing a
standard error naively from all frames will severely underestimate the true uncertainty.

**Block averaging** groups frames into blocks longer than the autocorrelation time $\tau$ so that
block means are approximately independent. The standard error is then estimated from the spread
of those block means.

Here we use it to get an error bar on the mean molecule count.

In [ ]:
from chempiler.core.statistics import block_average

# Estimate autocorrelation time from the molecule-count signal
# (in practice you'd use an autocorrelation function; here we pick a reasonable value)
tau_corr = 50   # frames — adjust based on your system's dynamics

mol_count_array = np.array(mol_counts, dtype=float)

stats = block_average(mol_count_array, tau_corr=tau_corr)

print(f"Block size (tau_corr) : {tau_corr} frames")
print(f"Number of blocks      : {stats['n_blocks']}")
print(f"Mean molecule count   : {stats['mean']:.4f} ± {stats['stderr']:.4f}")

In [ ]:
# Block-size convergence test: the estimated error should plateau once
# the block size exceeds the true autocorrelation time.
block_sizes = np.unique(np.geomspace(10, len(frames) // 3, num=20, dtype=int))
errors = []
for bs in block_sizes:
    try:
        s = block_average(mol_count_array, tau_corr=bs)
        errors.append(s['stderr'])
    except ValueError:
        errors.append(np.nan)

errors = np.array(errors)

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogx(block_sizes, errors, "o-", color="steelblue", lw=1.2, ms=4)
ax.set_xlabel("Block size (frames)")
ax.set_ylabel("Estimated std error")
ax.set_title("Block-size convergence — molecule count")
plt.tight_layout()
plt.show()

**Interpretation:** The estimated error rises with block size until it flattens into a plateau.
The plateau is the true standard error — use a block size anywhere in that plateau region.
If the curve keeps rising without flattening, the trajectory is too short to estimate statistics reliably.

---
## 8 — Custom tracking with the low-level `Tracker` API

All high-level functions (`atom_hop`, `ligand_exchange`, etc.) are thin wrappers around the
generic `Tracker` class. You can use `Tracker` directly to define completely custom state functions.

The API requires two callables:
- `entity_fn(frames)` → 1-D array of entity indices to observe (called once at startup)
- `state_fn(frame, entity)` → hashable state (called for every entity at every frame)

### Example: track which atoms are unbound

An atom is *unbound* when it is the only member of its molecule (singleton molecule).
Here we detect the frames when each oxygen transitions from bonded to free radical and back.

In [ ]:
from chempiler.core.tracker import Tracker
import numpy as np

tracker = Tracker(frames)

def oxygen_entities(frames):
    """Track all O atoms (indices are stable across frames)."""
    syms = frames[0].symbols
    return np.array([i for i, s in enumerate(syms) if s == "O"], dtype=np.int32)

def bonded_state(frame, atom_idx):
    """State is True if the O atom is in a molecule with >1 atom, False if isolated."""
    mol_id = frame.atom_to_mol[atom_idx]
    if mol_id < 0:
        return False
    return len(frame.molecules[mol_id]) > 1

bond_result = tracker.track(
    entity_fn=oxygen_entities,
    state_fn=bonded_state,
    persistence=2,   # require state to hold for 2 frames before recording
)

print(f"Bond-state change events: {bond_result['n_transitions']}")

if bond_result['n_transitions'] > 0:
    print("\nSample events (frame, O-atom, from, to):")
    for event in bond_result['transitions'][:5]:
        frame_idx, o_idx, from_s, to_s = event
        label = "bonded → free" if (not from_s and to_s) else "free → bonded" if (from_s and not to_s) else str((from_s, to_s))
        print(f"  frame {frame_idx:>5}  O[{o_idx}]  {label}")
else:
    print("All O atoms remained bonded throughout — expected for a stable water trajectory.")

### Example: track which H₂O molecules have more than 2 H-bond donors

Because `Tracker` accepts any hashable as a state, you can encode complex structural descriptors.
Here we count how many H atoms within 2.5 Å each O atom has (its total H-environment, including
hydrogen-bond donated H atoms as well as covalent ones).

In [ ]:
from chempiler.core.state_field import nearest_host

tracker2 = Tracker(frames)

def h_environment(frame, o_idx):
    """Count H atoms within 2.5 Å of this O (captures covalent + H-bond H)."""
    pos = frame.positions
    h_idx = np.array([i for i, s in enumerate(frame.symbols) if s == "H"], dtype=np.int32)
    if len(h_idx) == 0:
        return 0
    diffs = pos[h_idx] - pos[o_idx]
    if frame.atoms.get_pbc().any():
        cell = np.asarray(frame.atoms.get_cell())
        diffs -= np.round(diffs @ np.linalg.inv(cell)) @ cell
    dists = np.linalg.norm(diffs, axis=1)
    return int((dists < 2.5).sum())

env_result = tracker2.track(
    entity_fn=oxygen_entities,
    state_fn=h_environment,
    persistence=1,
)

print(f"H-environment change events: {env_result['n_transitions']}")

# Distribution of H-count values seen at the end of trajectory
# (residence_times here just reflects time spent in final state)
from collections import Counter
final_states = Counter(
    h_environment(frames[-1], o)
    for o in oxygen_entities(frames)
)
print("\nH-environment distribution in the last frame:")
for n_h, count in sorted(final_states.items()):
    print(f"  {count} O atoms have {n_h} H within 2.5 Å")

**Interpretation:** In liquid water each O atom typically has 2 covalent H neighbours and 2–3 H-bond
donated H neighbours, so an O-centred H count within 2.5 Å should peak around 4–5. The distribution
reflects the instantaneous H-bond network.

---
## Summary

| Task | Function / class |
|---|---|
| Load trajectory + cache | `ChempilerTrajectory(file).build(cache_file=...)` |
| Formula inventory | `traj.summary()` |
| Stable-composition segments | `segment_by_molecule_count(frames)` |
| RDF / coordination number | `traj.rdf(center, target, integrate=...)` |
| Proton / ion hopping | `atom_hop(frames, tracked, host, cutoff, persistence)` |
| Molecular identity changes | `ligand_exchange(frames)` |
| Bonding-environment changes | `coordination_dynamics(frames, atom_symbol)` |
| Statistical error bars | `block_average(values, tau_corr)` |
| Custom structural tracking | `Tracker(frames).track(entity_fn, state_fn, persistence)` |

All analyses work on standard Python lists of `Frame` objects, so you can slice, filter, or
combine frames freely before passing them to any function.